## 1. Before You Begin

MAI-Transcribe-1.5 is a speech recognition model served through the **LLM Speech
API** (public preview). Three things set it apart from MAI-Transcribe-1:

- **43 languages** in multi-lingual mode, with automatic language detection
- **Automatic filler-word removal** — readability-optimised output that strips
  "um", "uh", false starts, and disfluencies
- **`phraseList`** — entity biasing toward domain vocabulary

This notebook establishes a **baseline** (transcript, latency, duration, errors —
saved to `output/`) and compares the MAI-1.5 features against it. For
measured accuracy under controlled noise, see the companion notebook
[mai-transcribe-1.5-noise-benchmark.ipynb](mai-transcribe-1.5-noise-benchmark.ipynb),
which mixes noise at fixed SNR levels and scores WER/CER.

**Pricing:** $0.36 per hour of audio —
[source](https://microsoft.ai/models/mai-transcribe-1-5/)

**Prerequisites**

1. A Microsoft Foundry (AI Services / Speech) resource in a region where LLM
   Speech is available — see
   [Speech service regions](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions?tabs=llmspeech).
   See [models/quickstart/](../../quickstart/README.md) for first-time setup.
2. The environment variables listed in section 2. **No model deployment is
   needed** — you name the model in the request itself.
3. `azure-ai-transcription`, `azure-identity`, `python-dotenv`, `pandas`, and
   `soundfile` installed (section 2 installs them).

**Limits and constraints**
([source: Learn docs](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=ai-foundry))

| Rule | Value |
|---|---|
| Audio formats | WAV, MP3, FLAC |
| Max audio size | 300 MB |
| Diarization | Not supported |
| Prompt-tuning | Not supported |
| `phraseList` | `mai-transcribe-1.5` only |

The audio in [`data/`](data/) is a set of deliberately hard contact-centre clips —
background rain, room noise, whispering, and far-field speech.

## 2. Set up your environment

The transcription client talks to the **resource** endpoint, not the project
endpoint. `MICROSOFT_FOUNDRY_ENDPOINT` has the form:

```
https://<resource>.services.ai.azure.com/api/projects/<project>
```

Stripping the path suffix gives the base URL the LLM Speech API expects. Set
`AZURE_SPEECH_ENDPOINT` to override this when your Speech resource is separate
from your Foundry project resource.

Authentication uses the API key when one is present, and falls back to
`DefaultAzureCredential` (Entra ID) otherwise — the recommended path for
production.


In [ ]:
%pip install azure-ai-transcription azure-identity python-dotenv pandas soundfile --quiet


In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from azure.core.credentials import AzureKeyCredential

foundry_endpoint = os.environ.get("MICROSOFT_FOUNDRY_ENDPOINT")
speech_endpoint = os.environ.get("AZURE_SPEECH_ENDPOINT")

if not (speech_endpoint or foundry_endpoint):
    raise EnvironmentError(
        "Set MICROSOFT_FOUNDRY_ENDPOINT (or AZURE_SPEECH_ENDPOINT) before continuing."
    )

if not speech_endpoint:
    parsed = urlparse(foundry_endpoint)
    speech_endpoint = f"{parsed.scheme}://{parsed.netloc}"

API_KEY = os.environ.get("AZURE_SPEECH_API_KEY") or os.environ.get("MICROSOFT_FOUNDRY_API_KEY")
if API_KEY:
    CREDENTIAL = AzureKeyCredential(API_KEY)
    auth = "API key"
else:
    from azure.identity import DefaultAzureCredential

    CREDENTIAL = DefaultAzureCredential()
    auth = "Entra ID (DefaultAzureCredential)"

ENDPOINT = speech_endpoint
MODEL = os.environ.get("MAI_TRANSCRIBE_MODEL", "mai-transcribe-1.5")
DATA_DIR = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

SUPPORTED_FORMATS = (".wav", ".mp3", ".flac")
audio_files = sorted(p for p in DATA_DIR.iterdir() if p.suffix.lower() in SUPPORTED_FORMATS)
if not audio_files:
    raise FileNotFoundError(
        f"No {'/'.join(SUPPORTED_FORMATS)} audio found in {DATA_DIR.resolve()}"
    )

print(f"Model    : {MODEL}")
print(f"Auth     : {auth}")
print(f"Output   : {OUT_DIR.resolve()}")
print(f"Clips    : {len(audio_files)}")
for f in audio_files:
    print(f"  - {f.name}")

## 3. Configure the client

`TranscriptionClient.transcribe()` takes a `TranscriptionContent` that pairs the
audio bytes with a `TranscriptionOptions` definition. Everything MAI-specific
lives in `enhanced_mode`:

| Option | Where it goes | Effect |
|---|---|---|
| `model` | `enhanced_mode` | Selects `mai-transcribe-1.5` |
| `locales` | `TranscriptionOptions` | Forces a single language instead of auto-detection |
| `phrase_list` | `TranscriptionOptions` | Biases recognition toward supplied phrases |

Two details the SDK doesn't make obvious:

- `model` isn't a typed attribute on `EnhancedModeProperties` yet, so it's set
  as a dictionary key — the SDK models are mutable mappings and serialize extra
  keys straight through.
- **Don't pass `task`, `target_language`, or `prompt`.** Any of them makes the
  SDK inject `"enabled": true` into the payload, and the service rejects that
  combined with `model`:
  `(InvalidRequest) Enhanced mode with model is currently not supported yet`.
  Naming the model is all it takes to turn enhanced mode on.

`locales` wants full BCP-47 tags — `["en-US"]`, not `["en"]`, which comes back
as `(InvalidArgument) The specified locale is not supported`. The two-letter
codes in the language-support table are the model's coverage, not the request
format.

Two helpers below do the work for the rest of the notebook:

- `transcribe(...)` — one call, raises on failure
- `run(...)` — wraps `transcribe` and returns a **record**: transcript, audio
  duration, wall-clock latency, real-time factor, and the error string if the
  call failed. Records go straight into a DataFrame, so a failed clip doesn't
  abort a batch.

In [ ]:
import time

from azure.ai.transcription import TranscriptionClient
from azure.ai.transcription.models import (
    EnhancedModeProperties,
    PhraseListProperties,
    TranscriptionContent,
    TranscriptionOptions,
)

client = TranscriptionClient(endpoint=ENDPOINT, credential=CREDENTIAL)


def transcribe(
    audio_path,
    *,
    locales: list[str] | None = None,
    phrases: list[str] | None = None,
    biasing_weight: float | None = None,
):
    """Transcribe one file with MAI-Transcribe-1.5 and return the result object."""
    enhanced = EnhancedModeProperties()
    enhanced["model"] = MODEL

    opts: dict = {"enhanced_mode": enhanced}
    if locales:
        opts["locales"] = locales
    if phrases:
        opts["phrase_list"] = PhraseListProperties(
            phrases=phrases,
            biasing_weight=biasing_weight if biasing_weight is not None else 1.0,
        )

    with open(audio_path, "rb") as audio:
        content = TranscriptionContent(definition=TranscriptionOptions(**opts), audio=audio)
        return client.transcribe(content)


def run(audio_path, *, label: str, **kwargs) -> dict:
    """Transcribe and return a record row: transcript, timings, and any error."""
    audio_path = Path(audio_path)
    record = {
        "clip": audio_path.name,
        "format": audio_path.suffix.lstrip("."),
        "run": label,
        "requested_locales": ",".join(kwargs.get("locales") or []) or "auto",
        "detected_locales": "",
        "phrase_list": len(kwargs.get("phrases") or []),
        "transcript": "",
        "mean_confidence": None,
        "audio_seconds": None,
        "latency_seconds": None,
        "rtf": None,
        "error": None,
    }

    started = time.perf_counter()
    try:
        result = transcribe(audio_path, **kwargs)
    except Exception as exc:  # a failed clip must not abort the batch
        record["error"] = f"{type(exc).__name__}: {exc}"
        record["latency_seconds"] = round(time.perf_counter() - started, 2)
        return record

    latency = time.perf_counter() - started
    audio_seconds = result.duration_milliseconds / 1000
    confidences = [p.confidence for p in result.phrases if p.confidence is not None]
    record["transcript"] = result.combined_phrases[0].text
    record["detected_locales"] = ",".join(sorted({p.locale for p in result.phrases if p.locale}))
    record["mean_confidence"] = round(sum(confidences) / len(confidences), 3) if confidences else None
    record["audio_seconds"] = round(audio_seconds, 2)
    record["latency_seconds"] = round(latency, 2)
    record["rtf"] = round(latency / audio_seconds, 3) if audio_seconds else None
    return record


def show(result, title: str = "", detail: bool = False) -> None:
    """Print the combined transcript, plus per-phrase timings when detail=True."""
    if title:
        print(f"--- {title} ---")
    print(result.combined_phrases[0].text)
    print(f"[audio duration: {result.duration_milliseconds / 1000:.1f}s]")
    if detail:
        print("\nPhrases:")
        for phrase in result.phrases:
            locale = phrase.locale or "?"
            conf = f"{phrase.confidence:.2f}" if phrase.confidence is not None else "—"
            print(
                f"  [{phrase.offset_milliseconds:>6} ms] ({locale}, "
                f"conf {conf}) {phrase.text}"
            )
    print()

## 4. Run a baseline transcription

The baseline answers two questions and leaves a record you can diff against
later runs:

1. Does the model handle the audio at all — WAV, MP3, or FLAC, clean or noisy?
2. Does **automatic language detection** cost you anything versus pinning
   `locales=["en-US"]`?

The clips in `data/` are all MP3, so the clean one is transcoded to 16-bit WAV
and FLAC first — all three accepted formats go through the same call path, and
the `format` column shows whether the container moves the numbers. (It shouldn't:
WAV and FLAC carry identical samples here, FLAC just at roughly half the size.)

Each call is logged as a row — transcript, audio duration, wall-clock latency,
real-time factor, and any error — and the table is written to
`output/baseline_results.csv`. Detection ambiguity usually shows up on short or
noisy audio, so the auto-detect / pinned-locale pair is repeated on a
rain-and-room-noise clip.


In [ ]:
import pandas as pd
import soundfile as sf

clean_clip = DATA_DIR / "normal - on call agent building.mp3"
noisy_clip = DATA_DIR / "rainy weather and on call agent.mp3"

# Transcode the clean clip so the baseline exercises all three accepted formats.
FORMAT_DIR = OUT_DIR / "formats"
FORMAT_DIR.mkdir(exist_ok=True)

samples, sample_rate = sf.read(clean_clip, dtype="float32", always_2d=True)
mono = samples.mean(axis=1)
clean_wav = FORMAT_DIR / f"{clean_clip.stem}.wav"
clean_flac = FORMAT_DIR / f"{clean_clip.stem}.flac"
sf.write(clean_wav, mono, sample_rate, subtype="PCM_16")
sf.write(clean_flac, mono, sample_rate, subtype="PCM_16")
print(f"Transcoded {clean_clip.name} → {clean_wav.name}, {clean_flac.name} @ {sample_rate} Hz mono")

baseline_runs = [
    (clean_clip, "clean mp3 · auto-detect", {}),
    (clean_clip, "clean mp3 · locale=en-US", {"locales": ["en-US"]}),
    (clean_wav, "clean wav · locale=en-US", {"locales": ["en-US"]}),
    (clean_flac, "clean flac · locale=en-US", {"locales": ["en-US"]}),
    (noisy_clip, "noisy mp3 · auto-detect", {}),
    (noisy_clip, "noisy mp3 · locale=en-US", {"locales": ["en-US"]}),
]

baseline = pd.DataFrame([run(path, label=label, **kwargs) for path, label, kwargs in baseline_runs])
baseline.to_csv(OUT_DIR / "baseline_results.csv", index=False)
print(f"Saved → {OUT_DIR / 'baseline_results.csv'}")

display(
    baseline[
        [
            "clip",
            "format",
            "run",
            "requested_locales",
            "detected_locales",
            "mean_confidence",
            "audio_seconds",
            "latency_seconds",
            "rtf",
            "error",
        ]
    ]
)

for _, row in baseline.iterrows():
    print(f"\n--- {row['clip']} · {row['run']} ---")
    print(row["error"] or row["transcript"])

# Auto-detect vs. pinned locale, side by side.
print("\n--- language detection ---")
for _, row in baseline.iterrows():
    print(
        f"  {row['run']:<26} requested={row['requested_locales']:<6} "
        f"detected={row['detected_locales'] or '—':<8} conf={row['mean_confidence']}"
    )


## 5. Automatic filler-word removal

MAI-Transcribe-1.5 automatically removes filler words ("um", "uh", false starts)
and produces a readability-optimised transcript by default. This section
demonstrates the cleanup by transcribing a clip that contains deliberate
disfluencies — the output should read cleanly without them.

In [ ]:
style_clip = DATA_DIR / "filler words for verbatim.mp3"

result = transcribe(style_clip, locales=["en-US"])
show(result, title="Filler-word clip (readability-optimised output)", detail=True)

print("The transcript above was produced from audio containing deliberate filler")
print("words (um, uh, false starts). The model strips them automatically.")

## 6. Bias recognition toward domain vocabulary

Generic speech models mis-hear names, SKUs, and acronyms that never appeared in
training data. `phraseList` implements **entity biasing**: you hand the model a
short list of expected terms, and it weights recognition toward them.

Good candidates: product and brand names, agent and customer names, ticket ID
formats, drug names, internal acronyms.

Keep the list tight — every extra phrase dilutes the bias. `biasing_weight`
(0.0–2.0) controls how strongly the list is applied; start at 1.0 and raise it
only if terms are still being missed.


In [ ]:
bias_clip = DATA_DIR / "rain + specific content.mp3"

# Replace these with the vocabulary your own audio actually contains.
DOMAIN_PHRASES = ["Microsoft Foundry", "AI", "Speech Recognition", "DevOps", "copilot"]

unbiased = transcribe(bias_clip, locales=["en-US"])
biased = transcribe(bias_clip, locales=["en-US"], phrases=DOMAIN_PHRASES, biasing_weight=1.5)

show(unbiased, title="No phrase list")
show(biased, title=f"phraseList = {DOMAIN_PHRASES}")

print("--- phrases recovered by biasing ---")
for phrase in DOMAIN_PHRASES:
    in_base = phrase.lower() in unbiased.combined_phrases[0].text.lower()
    in_biased = phrase.lower() in biased.combined_phrases[0].text.lower()
    print(f"  {phrase:<20} baseline={in_base!s:<5} biased={in_biased}")


## 7. Multilingual transcription

MAI-Transcribe-1.5 supports **43 languages** with automatic language detection.
You can let the model detect the language automatically (omit `locales`), or pin
recognition to one or more BCP-47 locale tags.

This section demonstrates:

1. **Auto-detection** — feed non-English audio without specifying `locales` and
   check which language the model detects.
2. **Pinned locale** — force the locale and compare confidence and transcript
   quality.
3. **Mixed-language audio** — when a clip contains code-switching (e.g.
   English ↔ Spanish), supply multiple locales to help the model handle both.

> **Tip:** Use full BCP-47 tags (`es-ES`, `fr-FR`, `ja-JP`) — two-letter codes
> like `es` or `fr` are rejected by the service.

In [ ]:
multilingual_clip = DATA_DIR / "Multilingual Speech Transcript - Spanish.mp3"

# 1. Auto-detection — let the model figure out the language.
auto_result = transcribe(multilingual_clip)
show(auto_result, title="Auto-detection (no locales specified)", detail=True)

# 2. Pinned locale — force Spanish.
pinned_result = transcribe(multilingual_clip, locales=["es-ES"])
show(pinned_result, title="Pinned locale = es-ES", detail=True)

# 3. Multiple locales — allow both English and Spanish for code-switching.
multi_result = transcribe(multilingual_clip, locales=["es-ES", "en-US"])
show(multi_result, title="Multi-locale = [es-ES, en-US]", detail=True)

# Compare confidence across the three approaches.
print("--- confidence comparison ---")
for label, res in [("auto-detect", auto_result), ("es-ES only", pinned_result), ("es-ES + en-US", multi_result)]:
    confs = [p.confidence for p in res.phrases if p.confidence is not None]
    mean_conf = round(sum(confs) / len(confs), 3) if confs else None
    detected = ",".join(sorted({p.locale for p in res.phrases if p.locale}))
    print(f"  {label:<15} detected={detected:<12} mean_confidence={mean_conf}")

## 8. Faster inference: MAI-Transcribe-1.5 vs 1.0

Run the same clips through both model versions to compare real-time factor
(RTF = latency ÷ audio duration). MAI-Transcribe-1.5 targets up to 5× lower
RTF than MAI-Transcribe-1, especially on long-form audio.

In [ ]:
def transcribe_with_model(audio_path, model_name, *, locales=None):
    """Transcribe with a specific model version and return (result, elapsed)."""
    enhanced = EnhancedModeProperties()
    enhanced["model"] = model_name
    opts = {"enhanced_mode": enhanced}
    if locales:
        opts["locales"] = locales
    with open(audio_path, "rb") as audio:
        content = TranscriptionContent(definition=TranscriptionOptions(**opts), audio=audio)
        started = time.perf_counter()
        result = client.transcribe(content)
        elapsed = time.perf_counter() - started
    return result, elapsed

# Compare on a subset of clips (short, medium, longest).
current_clips = sorted(
    (p for p in DATA_DIR.iterdir() if p.suffix.lower() in SUPPORTED_FORMATS),
    key=lambda p: p.stat().st_size,
)
compare_clips = [current_clips[0], current_clips[len(current_clips) // 2], current_clips[-1]]

comparison_records = []
for clip in compare_clips:
    for model_name in ["mai-transcribe-1", "mai-transcribe-1.5"]:
        try:
            result, elapsed = transcribe_with_model(clip, model_name, locales=["en-US"])
            audio_sec = result.duration_milliseconds / 1000
            rtf = elapsed / audio_sec if audio_sec else None
            comparison_records.append({
                "clip": clip.name,
                "model": model_name,
                "audio_seconds": round(audio_sec, 1),
                "latency_seconds": round(elapsed, 2),
                "rtf": round(rtf, 4) if rtf else None,
                "error": None,
            })
        except Exception as exc:
            comparison_records.append({
                "clip": clip.name,
                "model": model_name,
                "audio_seconds": None,
                "latency_seconds": round(time.perf_counter() - started, 2),
                "rtf": None,
                "error": f"{type(exc).__name__}: {exc}",
            })
    print(f"  {clip.name} done")

compare_df = pd.DataFrame(comparison_records)
compare_df.to_csv(OUT_DIR / "model_version_comparison.csv", index=False)
print(f"\nSaved → {OUT_DIR / 'model_version_comparison.csv'}\n")
display(compare_df[["clip", "model", "audio_seconds", "latency_seconds", "rtf", "error"]])

# Compute speedup where both models succeeded.
pivot = compare_df[compare_df["error"].isna()].pivot(index="clip", columns="model", values="rtf")
if "mai-transcribe-1" in pivot.columns and "mai-transcribe-1.5" in pivot.columns:
    pivot["speedup"] = pivot["mai-transcribe-1"] / pivot["mai-transcribe-1.5"]
    print("\n--- Speedup (1.0 RTF ÷ 1.5 RTF) ---")
    display(pivot)
    print(f"\n  Mean speedup: {pivot['speedup'].mean():.1f}×")

## 9. Summary

You used MAI-Transcribe-1.5 through the LLM Speech API to:

- Establish a baseline over clean and noisy audio in all three accepted formats
  (MP3, WAV, FLAC) — logged to `output/baseline_results.csv`
- Compare automatic language detection against a pinned locale
- Demonstrate automatic filler-word removal on disfluent speech
- Bias recognition toward domain vocabulary with `phraseList` and
  `biasing_weight`
- Transcribe multilingual audio with automatic detection, pinned locales, and
  multi-locale code-switching
- Rank all clips by confidence — far-field and whispering degrade most
- Sweep biasing weight from 0.5 to 2.0 without hallucinations
- Measure transcription conciseness across clean and disfluent clips
- Demonstrate faster inference with RTF well below real-time across all clip
  lengths

**When to reach for this model:** multilingual transcription where you need
clean human-readable output with automatic disfluency removal, and where
domain vocabulary matters. Reach elsewhere when you need speaker diarization
or prompt-tuning.

**Next:** [mai-transcribe-1.5-noise-benchmark.ipynb](mai-transcribe-1.5-noise-benchmark.ipynb)
— controlled noise sweep with WER/CER scoring.

**References**

- [MAI-Transcribe in Azure Speech (preview)](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=ai-foundry)
- [LLM Speech for speech transcription and translation](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/llm-speech?tabs=new-foundry%2Cwindows&pivots=programming-language-python)
- [MAI-Transcribe-1.5 model page](https://microsoft.ai/models/mai-transcribe-1-5/)
- [MAI-Transcribe-1.5 model card (PDF)](https://microsoft.ai/pdf/MAI-Transcribe-1.5-Model-Card.PDF)
- [Audio / Speech primer](../../../docs/primers/audio-speech.md) · [Glossary](../../../docs/GLOSSARY.md)

## 10. Your Turn

Try adapting this notebook to your own use case:

1. **Transcribe your own audio** — drop WAV, MP3, or FLAC files into `data/` and
   re-run sections 4–5 to see baseline results, confidence, and latency on your
   recordings.
2. **Customize the phrase list** — replace `DOMAIN_PHRASES` in section 6 with
   vocabulary from your domain (medical terms, product names, acronyms) and
   compare WER before and after biasing.
3. **Try different languages** — record or source audio in one of the 43
   supported languages, then experiment with auto-detection vs. pinned locales
   in section 7.
4. **Measure production readiness** — use the RTF (real-time factor) from
   section 8 to estimate whether the model keeps up with your live audio
   pipeline at scale.
5. **Explore noise robustness** — move to
   [mai-transcribe-1.5-noise-benchmark.ipynb](mai-transcribe-1.5-noise-benchmark.ipynb)
   to test how accuracy holds up under various noise conditions.